# CycleGAN portfolio browsing copy
Derived from the executed Task3_CycleGAN_Naveen_Repro.ipynb. All outputs and execution counts were cleared; smoke/training switches were disabled. Model architecture and training logic are unchanged. This copy has not been executed.
Read setup and evaluation cells before running; inference/evaluation cells still require local artifacts and can write files. The recorded run uses seed 171; src/config.py uses 266 and is not an exact substitute.


# DATA266 Task 3 — CycleGAN reproduction, Naveen

Execute cells **manually, in order, in VS Code** using the `data266-task3` Python kernel. This notebook has never been executed. Seed **171**, SID4 **171**. A = Monet; B = Photo; A2B generates photos and B2A generates Monet-style images.

All file-writing helpers enforce the separate `naveen/checkpoints_repro/`, `naveen/logs_repro/`, and `naveen/outputs_repro/` roots. Each manual run gets a unique subdirectory. Existing official, pilot, and evaluation files are read-only references and are never used as reproduction checkpoints.

The architecture and loss/training-step definitions below are copied from the existing implementation. Batch size is 1, one epoch is 300 steps, each Monet appears once, and photos are drawn uniformly with replacement. The scheduler keeps epochs 1–100 at 2e-4 and retains the original 101–200 decay definition. The manual training entry point defaults to stopping at epoch 100.

**Smoke testing and training are disabled by default.** Cell 27 defines the loop. Cells 28–31 define its helpers. Only after inspecting the setup should you manually enable the explicit switch in cell 26 (smoke test) or cell 31 (training). Running the notebook as delivered does not start either training workload. No inference, instructor evaluation, or Kaggle submission is automatically launched.


## PHASE 1 — Environment and data


### 1. Environment/GPU

Inspect the selected kernel, versions, CUDA runtime, GPU, VRAM, OS, and driver. This cell does not install packages.


In [ ]:
import sys
import platform
import subprocess
import importlib.metadata
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENVIRONMENT = {
    "python": sys.version,
    "pytorch": torch.__version__,
    "torchvision": importlib.metadata.version("torchvision"),
    "cuda_runtime": torch.version.cuda,
    "os": platform.platform(),
    "device": str(DEVICE),
    "gpu_name": None,
    "gpu_vram_GiB": None,
    "nvidia_driver": None,
}
if DEVICE.type == "cuda":
    gpu = torch.cuda.get_device_properties(DEVICE)
    ENVIRONMENT["gpu_name"] = gpu.name
    ENVIRONMENT["gpu_vram_GiB"] = gpu.total_memory / 1024**3
try:
    ENVIRONMENT["nvidia_driver"] = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        text=True, timeout=10,
    ).strip()
except (OSError, subprocess.SubprocessError):
    pass
for name, value in ENVIRONMENT.items():
    print(f"{name}: {value}")


### 2. Imports

Self-contained notebook imports. No official training, inference, or evaluation entry point is imported.


In [ ]:
import csv
import json
import math
import random
import time
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm


### 3. Seed

Set the explicitly requested seed and SID4. Fresh reproduction training resets this seed before initialization; exploration and smoke testing cannot contaminate a fresh run.


In [ ]:
SEED = 171
SID4 = int("0171")
assert SEED == SID4 == 171

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def random_states():
    return dict(
        python=random.getstate(), numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )

def restore_random(states):
    random.setstate(states["python"])
    np.random.set_state(states["numpy"])
    torch.set_rng_state(states["torch"].cpu())
    if states["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([state.cpu() for state in states["cuda"]])

seed_all(SEED)
print(f"SEED={SEED}; SID4={SID4}")


### 4. Paths

Find the project relative to the current notebook working directory. Merely inspecting paths creates no folders. All later writes pass the reproduction-only path guard.


In [ ]:
ROOT = next(
    (candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (candidate / "data/monet_jpg").is_dir()
     and (candidate / "data/photo_jpg").is_dir()
     and (candidate / "src").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open VS Code in the project root or a directory within it.")

DATA_A = ROOT / "data/monet_jpg"
DATA_B = ROOT / "data/photo_jpg"
CHECKPOINTS_REPRO = ROOT / "checkpoints_repro"
LOGS_REPRO = ROOT / "logs_repro"
OUTPUTS_REPRO = ROOT / "outputs_repro"
REPRO_ROOTS = (CHECKPOINTS_REPRO, LOGS_REPRO, OUTPUTS_REPRO)

def require_repro_path(path, specific_root=None):
    path = Path(path).resolve()
    roots = (specific_root,) if specific_root is not None else REPRO_ROOTS
    if not any(path.is_relative_to(Path(root).resolve()) for root in roots):
        raise PermissionError(f"Refusing access outside reproduction artifact roots: {path}")
    return path

def write_repro_json(path, value):
    path = require_repro_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2)
        handle.write("\n")

def save_repro_image(tensor, path):
    path = require_repro_path(path, OUTPUTS_REPRO)
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite an existing image: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    array = ((tensor.detach().cpu().squeeze(0).clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
    Image.fromarray(array.permute(1, 2, 0).numpy()).convert("RGB").save(path)
    with Image.open(path) as image:
        assert image.mode == "RGB" and image.size == (256, 256)
        image.verify()

def new_run_id():
    return "repro_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid.uuid4().hex[:8]

for name, path in [("Monet", DATA_A), ("Photo", DATA_B),
                   ("checkpoints", CHECKPOINTS_REPRO), ("logs", LOGS_REPRO), ("outputs", OUTPUTS_REPRO)]:
    print(name, path.relative_to(ROOT).as_posix())


### 5. Dataset counts

Read the existing JPEG lists in stable filename order. No dataset copies or output files are created.


In [ ]:
PATHS_A = sorted(path for path in DATA_A.iterdir() if path.suffix.lower() in (".jpg", ".jpeg"))
PATHS_B = sorted(path for path in DATA_B.iterdir() if path.suffix.lower() in (".jpg", ".jpeg"))
assert len(PATHS_A) == 300, f"Expected 300 Monet images, found {len(PATHS_A)}"
assert len(PATHS_B) == 7038, f"Expected 7038 Photo images, found {len(PATHS_B)}"
assert len({path.stem for path in PATHS_A}) == 300
assert len({path.stem for path in PATHS_B}) == 7038
print(f"A = Monet: {len(PATHS_A)}; B = Photo: {len(PATHS_B)}")


### 6. Image verification

Check every source JPEG for valid encoding, 256×256 resolution, and RGB mode. These checks only read the dataset.


In [ ]:
for domain, paths in [("A / Monet", PATHS_A), ("B / Photo", PATHS_B)]:
    for path in tqdm(paths, desc=f"Verify {domain}"):
        with Image.open(path) as image:
            assert image.size == (256, 256), f"Unexpected dimensions: {path.name}"
            assert image.mode == "RGB", f"Unexpected image mode: {path.name}"
            image.verify()
    print(f"{domain}: {len(paths)} valid 256x256 RGB JPEGs")


### 7. Dataset visualization

Display four deterministically chosen real examples per domain. A local RNG preserves the training RNG stream; no images are selected based on model quality.


In [ ]:
preview_rng = random.Random(SEED)
preview = {"A": preview_rng.sample(PATHS_A, 4), "B": preview_rng.sample(PATHS_B, 4)}
figure, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, (domain, title) in enumerate([("A", "Monet"), ("B", "Photo")]):
    for column, path in enumerate(preview[domain]):
        with Image.open(path) as image:
            axes[row, column].imshow(image.convert("RGB"))
        axes[row, column].set_title(f"{domain}: {title}\n{path.name}")
        axes[row, column].axis("off")
figure.tight_layout()
plt.show()


### 8. Configuration

Preserve the existing architecture, optimizer, augmentation, losses, initialization, precision, and full LR schedule. Artifact paths and the requested reproduction seed/SID4 are recorded explicitly.


In [ ]:
CONFIG = dict(domain_A="Monet", domain_B="Photo", data_A="data/monet_jpg",
              data_B="data/photo_jpg", output_root="outputs_repro", image_size=256,
              resize_size=286, residual_blocks=9, generator_channels=64,
              discriminator_channels=64, normalization="InstanceNorm",
              output_activation="Tanh", discriminator="70x70 PatchGAN",
              adversarial="LSGAN MSE", lambda_cycle=10.0, lambda_identity=5.0,
              optimizer="Adam", learning_rate=2e-4, betas=(0.5, 0.999),
              batch_size=1, epochs=200, steps_per_epoch=300, constant_epochs=100,
              seed=SEED, fixed_sample_count=4, precision="float32",
              initialization="normal(0, 0.02)", replay_buffer=False,
              sampling="Monet permutation; independent uniform Photo with replacement",
              augmentation=["Resize 286", "RandomCrop 256", "RandomHorizontalFlip",
                            "ToTensor", "Normalize mean/std 0.5"],
              lr_schedule="epochs 1-100: 2e-4; epochs 101-200: 2e-4*(200-epoch)/100")
CONFIG["sid4"] = SID4
CONFIG["artifact_paths"] = {
    "checkpoints": "checkpoints_repro",
    "logs": "logs_repro",
    "outputs": "outputs_repro",
}
assert CONFIG["batch_size"] == 1 and CONFIG["steps_per_epoch"] == 300
assert CONFIG["residual_blocks"] == 9
assert CONFIG["lambda_cycle"] == 10 and CONFIG["lambda_identity"] == 5
print(json.dumps(CONFIG, indent=2))
print("Manual training stop defaults to epoch 100; the unchanged schedule is defined through epoch 200.")


### 9. Transforms

Training: Resize 286, RandomCrop 256, RandomHorizontalFlip, ToTensor, Normalize to [-1,1]. Evaluation: deterministic Resize 256, ToTensor, Normalize. These definitions are copied from the existing dataset module.


In [ ]:
def transform(training=False):
    ops = ([transforms.Resize((286, 286)), transforms.RandomCrop(256),
            transforms.RandomHorizontalFlip()] if training else
           [transforms.Resize((256, 256))])
    return transforms.Compose(ops + [transforms.ToTensor(),
                                    transforms.Normalize((.5,)*3, (.5,)*3)])

def load_image(path, training=False):
    with Image.open(path) as im:
        return transform(training)(im.convert("RGB"))
train_transform = transform(training=True)
eval_transform = transform(training=False)
print("Training:", train_transform)
print("Evaluation:", eval_transform)


### 10. Dataset class

The existing EpochDataset generates one Monet permutation and independent uniform Photo draws with replacement. Photos may repeat; Monet images cannot repeat within an epoch.


In [ ]:
class EpochDataset(Dataset):
    def __init__(self, paths_A, paths_B, tiny=False):
        self.paths_A, self.paths_B = paths_A, paths_B
        indices = random.sample(range(len(paths_A)), len(paths_A))
        if tiny:
            indices = indices[:2]
        self.pairs = [(i, random.randrange(len(paths_B))) for i in indices]
        assert len(set(i for i, _ in self.pairs)) == len(self.pairs)
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, index):
        a, b = self.pairs[index]
        return load_image(self.paths_A[a], True), load_image(self.paths_B[b], True)
    def manifest(self):
        return [{"step": n+1, "A": self.paths_A[a].relative_to(ROOT).as_posix(),
                 "B": self.paths_B[b].relative_to(ROOT).as_posix()}
                for n, (a, b) in enumerate(self.pairs)]


### 11. DataLoader

Preview the 300-pair sampling plan without running model computation. A new plan is constructed every training epoch; the fresh training entry point resets seed 171.


In [ ]:
preview_dataset = EpochDataset(PATHS_A, PATHS_B)
assert len(preview_dataset) == 300
assert sorted(index_a for index_a, _ in preview_dataset.pairs) == list(range(300))
assert all(0 <= index_b < 7038 for _, index_b in preview_dataset.pairs)
preview_loader = DataLoader(preview_dataset, batch_size=1, shuffle=False, num_workers=0)
assert len(preview_loader) == 300
print("300 steps; batch size 1; each Monet once; independent uniform Photo sampling with replacement.")
print("First five pairs:", preview_dataset.manifest()[:5])


## PHASE 2 — Models

The following architecture definitions are copied verbatim from the existing source. No pretrained models are loaded.


### 12. InstanceNorm helper

InstanceNorm2d with affine=False and track_running_stats=False, matching the official source.


In [ ]:
def norm(channels):
    return nn.InstanceNorm2d(channels, affine=False, track_running_stats=False)


### 13. Residual block

Reflection padding, two 3×3 convolutions, InstanceNorm, ReLU, and the residual addition.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
                                 norm(channels), nn.ReLU(True), nn.ReflectionPad2d(1),
                                 nn.Conv2d(channels, channels, 3), norm(channels))
    def forward(self, x):
        return x + self.net(x)


### 14. ResNet-9 generator

7×7 RGB stem, two downsampling stages, nine residual blocks, two transposed-convolution upsampling stages, 7×7 RGB output, Tanh.


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [nn.ReflectionPad2d(3), nn.Conv2d(3, 64, 7), norm(64), nn.ReLU(True)]
        for c in (64, 128):
            layers += [nn.Conv2d(c, c*2, 3, stride=2, padding=1), norm(c*2), nn.ReLU(True)]
        layers += [ResidualBlock(256) for _ in range(9)]
        for c in (256, 128):
            layers += [nn.ConvTranspose2d(c, c//2, 3, stride=2, padding=1,
                                         output_padding=1), norm(c//2), nn.ReLU(True)]
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(64, 3, 7), nn.Tanh()]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


### 15. 70×70 PatchGAN discriminator

Spatial real/fake prediction map. The first layer has no normalization; subsequent feature layers use InstanceNorm and LeakyReLU.


In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [nn.Conv2d(3, 64, 4, stride=2, padding=1), nn.LeakyReLU(.2, True)]
        for ci, co, stride in ((64, 128, 2), (128, 256, 2), (256, 512, 1)):
            layers += [nn.Conv2d(ci, co, 4, stride=stride, padding=1), norm(co),
                       nn.LeakyReLU(.2, True)]
        layers += [nn.Conv2d(512, 1, 4, stride=1, padding=1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


### 16. Weight initialization

Normal(0,0.02) convolution weights and zero biases, identical to the existing implementation.


In [ ]:
def initialize(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, 0., .02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


### 17. Build four models

Create fresh notebook-only G_A2B, G_B2A, D_A, and D_B. Official checkpoints are never loaded here.


In [ ]:
def build_models(device):
    models = dict(G_A2B=Generator(), G_B2A=Generator(), D_A=Discriminator(), D_B=Discriminator())
    for model in models.values():
        model.apply(initialize)
        model.to(device)
    return models
preview_models = build_models(DEVICE)
print("Fresh models:", list(preview_models))


### 18. Parameter counts

Inspect all four model sizes and confirm the copied architecture's expected counts.


In [ ]:
parameter_counts = {name: sum(p.numel() for p in model.parameters())
                    for name, model in preview_models.items()}
assert parameter_counts["G_A2B"] == parameter_counts["G_B2A"] == 11378179
assert parameter_counts["D_A"] == parameter_counts["D_B"] == 2764737
display(pd.DataFrame.from_dict(parameter_counts, orient="index", columns=["parameters"]))


### 19. Shape verification

Forward-only shape checks on synthetic tensors: generator 1×3×256×256 and discriminator 1×1×30×30. Fresh training later reinitializes the seed and models.


In [ ]:
with torch.no_grad():
    shape_input = torch.randn(1, 3, 256, 256, device=DEVICE)
    shape_report = {}
    for name, model in preview_models.items():
        result = model(shape_input)
        expected = (1, 3, 256, 256) if name.startswith("G_") else (1, 1, 30, 30)
        assert tuple(result.shape) == expected
        assert bool(torch.isfinite(result).all())
        shape_report[name] = list(result.shape)
    del shape_input, result
print(shape_report)


## PHASE 3 — Losses, optimizers, and manual smoke test


### 20. LSGAN, cycle, and identity losses

MSE adversarial loss; separately weighted L1 cycle losses (10 each) and L1 identity losses (5 each). Logged cycle/identity values include these weights. Definitions are copied from the existing losses module.


In [ ]:
MSE = nn.MSELoss()
L1 = nn.L1Loss()
def adversarial(prediction, real):
    return MSE(prediction, torch.ones_like(prediction) if real else torch.zeros_like(prediction))

def generator_losses(models, a, b):
    ga, gb = models["G_A2B"], models["G_B2A"]
    fake_b, fake_a = ga(a), gb(b)
    rec_a, rec_b = gb(fake_b), ga(fake_a)
    identity_b, identity_a = ga(b), gb(a)
    losses = dict(G_A2B_adv=adversarial(models["D_B"](fake_b), True),
                  G_B2A_adv=adversarial(models["D_A"](fake_a), True),
                  cycle_A=CONFIG["lambda_cycle"]*L1(rec_a, a),
                  cycle_B=CONFIG["lambda_cycle"]*L1(rec_b, b),
                  identity_A=CONFIG["lambda_identity"]*L1(identity_a, a),
                  identity_B=CONFIG["lambda_identity"]*L1(identity_b, b))
    losses["G_total"] = sum(losses.values())
    tensors = dict(real_A=a, real_B=b, fake_A=fake_a, fake_B=fake_b,
                   reconstructed_A=rec_a, reconstructed_B=rec_b,
                   identity_A=identity_a, identity_B=identity_b)
    return losses, tensors

def discriminator_loss(model, real, fake):
    return .5*(adversarial(model(real), True) + adversarial(model(fake.detach()), False))
LOSS_KEYS = ["G_A2B_adv", "G_B2A_adv", "cycle_A", "cycle_B",
             "identity_A", "identity_B", "G_total", "D_A", "D_B"]


### 21. Adam optimizers

One shared generator optimizer and separate D_A/D_B optimizers, matching the official setup. Every optimizer uses lr=2e-4 and betas=(0.5,0.999).


In [ ]:
def make_optimizers(models):
    gp = list(models["G_A2B"].parameters()) + list(models["G_B2A"].parameters())
    dp = list(models["D_A"].parameters()) + list(models["D_B"].parameters())
    optimizers = {
        "G": torch.optim.Adam(gp, lr=CONFIG["learning_rate"], betas=CONFIG["betas"]),
        "D_A": torch.optim.Adam(models["D_A"].parameters(), lr=CONFIG["learning_rate"], betas=CONFIG["betas"]),
        "D_B": torch.optim.Adam(models["D_B"].parameters(), lr=CONFIG["learning_rate"], betas=CONFIG["betas"]),
    }
    return optimizers, gp, dp

preview_optimizers, preview_gp, preview_dp = make_optimizers(preview_models)
for name, optimizer in preview_optimizers.items():
    print(name, optimizer.param_groups[0]["lr"], optimizer.param_groups[0]["betas"])


### 22. LR scheduler

The scheduler index 0 corresponds to epoch 1. Epochs 1–100 remain at 2e-4; epoch 101 is 0.000198 and epoch 200 is zero. The epoch-100 run boundary preserves the current LR and records a pending scheduler step for explicit later resume.


In [ ]:
def epoch_lr(epoch):
    if not 1 <= epoch <= 200:
        raise ValueError("Epoch must be in 1..200")
    return CONFIG["learning_rate"] * (1.0 if epoch <= 100 else (200 - epoch) / 100)

def make_schedulers(optimizers):
    return {
        name: torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda i: 1.0 if i < 100 else max(0.0, (199 - i) / 100)
        )
        for name, optimizer in optimizers.items()
    }

preview_schedulers = make_schedulers(preview_optimizers)
for epoch in (1, 100, 101, 150, 200):
    print(f"Epoch {epoch}: expected LR {epoch_lr(epoch):.8f}")


### 23. Replay-buffer policy

The official implementation used no image/replay buffer. Preserve that policy; no replay pool is introduced.


In [ ]:
USE_REPLAY_BUFFER = False
assert CONFIG["replay_buffer"] is False and USE_REPLAY_BUFFER is False
print("Replay buffer disabled, matching the official implementation.")


### 24. Gradient norm utility

Global L2 norms before optimizer updates, plus NaN/Inf counts over gradient elements. This definition is copied from the existing utility module.


In [ ]:
def gradient_stats(parameters):
    grads = [p.grad.detach() for p in parameters if p.grad is not None]
    nan = sum(int(torch.isnan(g).sum()) for g in grads)
    inf = sum(int(torch.isinf(g).sum()) for g in grads)
    norm = float(torch.sqrt(sum(g.float().square().sum() for g in grads))) if grads else 0.
    return norm, nan, inf


### 25. One training-step function

Train both cycle paths, both identity paths, both generators, and both discriminators. The finite-value guards and update order are copied from the official training source. A failure stops execution; the reproduction loop later saves diagnostic state.


In [ ]:
class NonfiniteTrainingError(FloatingPointError):
    def __init__(self, phase, losses, tensors, nan, inf, generator_updated=False):
        super().__init__(f"Nonfinite training values during {phase}: NaN={nan}, Inf={inf}")
        self.diagnostics = dict(phase=phase, losses=losses, NaN_count=nan, Inf_count=inf,
                                generator_updated=generator_updated,
                                tensors={k: v.detach().cpu() for k, v in tensors.items()})

def train_step(models, optimizers, gp, dp, a, b):
    for p in dp:
        p.requires_grad_(False)
    optimizers["G"].zero_grad(set_to_none=True)
    losses, tensors = generator_losses(models, a, b)
    scalars = {k: float(v.detach()) for k, v in losses.items()}
    nan_l = sum(math.isnan(v) for v in scalars.values())
    inf_l = sum(math.isinf(v) for v in scalars.values())
    if nan_l or inf_l:
        raise NonfiniteTrainingError("generator losses", scalars, tensors, nan_l, inf_l)
    losses["G_total"].backward()
    gn, nan_g, inf_g = gradient_stats(gp)
    if nan_g or inf_g or not math.isfinite(gn):
        raise NonfiniteTrainingError("generator gradients", scalars, tensors,
                                     nan_g+int(math.isnan(gn)), inf_g+int(math.isinf(gn)))
    optimizers["G"].step()
    for p in dp:
        p.requires_grad_(True)
    for domain in ("A", "B"):
        key = "D_" + domain
        optimizers[key].zero_grad(set_to_none=True)
        losses[key] = discriminator_loss(models[key], tensors["real_"+domain], tensors["fake_"+domain])
        value = float(losses[key].detach())
        scalars[key] = value
        if not math.isfinite(value):
            raise NonfiniteTrainingError(key+" loss", scalars, tensors,
                                         int(math.isnan(value)), int(math.isinf(value)), True)
        losses[key].backward()
    dn, nan_d, inf_d = gradient_stats(dp)
    if nan_d or inf_d or not math.isfinite(dn):
        raise NonfiniteTrainingError("discriminator gradients", scalars, tensors,
                                     nan_d+int(math.isnan(dn)), inf_d+int(math.isinf(dn)), True)
    optimizers["D_A"].step()
    optimizers["D_B"].step()
    scalars.update(generator_gradient_norm=gn, discriminator_gradient_norm=dn,
                   NaN_count=nan_g+nan_d+nan_l, Inf_count=inf_g+inf_d+inf_l)
    return scalars, tensors


### 26. Two-step smoke test — disabled by default

Only set RUN_SMOKE_TEST_REPRO=True when you manually want to run it. This creates fresh private models, performs exactly two complete training steps, checks both cycle paths, prints shapes and losses, saves temporary samples/checkpoint, verifies reload, and restores the notebook RNG state. It cannot overwrite training checkpoints or logs.


In [ ]:
def smoke_test_repro():
    saved_rng = random_states()
    folder = require_repro_path(OUTPUTS_REPRO / "audit" / ("smoke_" + uuid.uuid4().hex))
    folder.mkdir(parents=True, exist_ok=False)
    try:
        seed_all(SEED)
        models = build_models(DEVICE)
        optimizers, gp, dp = make_optimizers(models)
        schedulers = make_schedulers(optimizers)
        dataset = EpochDataset(PATHS_A[:2], PATHS_B[:2], tiny=True)
        assert len(dataset) == 2
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats(DEVICE)
        records = []
        for index, (a, b) in enumerate(DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0), 1):
            values, tensors = train_step(models, optimizers, gp, dp, a.to(DEVICE), b.to(DEVICE))
            assert all(math.isfinite(value) for value in values.values())
            shapes = {name: list(tensor.shape) for name, tensor in tensors.items()}
            assert all(shape == [1, 3, 256, 256] for shape in shapes.values())
            with torch.no_grad():
                shapes["D_A"] = list(models["D_A"](tensors["real_A"]).shape)
                shapes["D_B"] = list(models["D_B"](tensors["real_B"]).shape)
            assert shapes["D_A"] == shapes["D_B"] == [1, 1, 30, 30]
            print(f"Smoke step {index}/2:", values)
            print("Shapes:", shapes)
            if index == 1:
                save_repro_image(tensors["fake_A"], folder / "temporary_B2A.png")
                save_repro_image(tensors["fake_B"], folder / "temporary_A2B.png")
            records.append(values)
            del tensors
        assert len(records) == 2
        payload = {
            "run_kind": "repro_smoke", "epoch": 1, "global_step": 2,
            "configuration": CONFIG,
            "models": {name: model.state_dict() for name, model in models.items()},
            "optimizers": {name: opt.state_dict() for name, opt in optimizers.items()},
            "schedulers": {name: scheduler.state_dict() for name, scheduler in schedulers.items()},
            "random_states": random_states(),
        }
        checkpoint_path = require_repro_path(folder / "temporary.pt")
        torch.save(payload, checkpoint_path)
        expected_rng = (random.random(), torch.rand(3))
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        for name, model in models.items():
            with torch.no_grad():
                next(model.parameters()).add_(1.0)
            model.load_state_dict(checkpoint["models"][name])
            assert all(torch.equal(value.detach().cpu(), checkpoint["models"][name][key])
                       for key, value in model.state_dict().items())
        for name, optimizer in optimizers.items():
            optimizer.load_state_dict(checkpoint["optimizers"][name])
            schedulers[name].load_state_dict(checkpoint["schedulers"][name])
        restore_random(checkpoint["random_states"])
        assert random.random() == expected_rng[0] and torch.equal(torch.rand(3), expected_rng[1])
        peak = torch.cuda.max_memory_allocated(DEVICE) / 1024**3 if DEVICE.type == "cuda" else 0.0
        report = {
            "seed": SEED, "sid4": SID4, "steps": 2, "shapes": shapes,
            "parameter_counts": {name: sum(p.numel() for p in model.parameters())
                                 for name, model in models.items()},
            "losses": records, "peak_gpu_memory_GiB": peak, "checkpoint_reload_succeeded": True,
        }
        write_repro_json(folder / "smoke_report.json", report)
        print("Checkpoint reload passed; peak allocated GPU memory:", peak, "GiB")
        print("Smoke artifacts:", folder.relative_to(ROOT))
        return report
    finally:
        restore_random(saved_rng)

RUN_SMOKE_TEST_REPRO = False
if RUN_SMOKE_TEST_REPRO:
    SMOKE_REPORT_REPRO = smoke_test_repro()
else:
    print("Smoke test is disabled. Enable this cell's switch manually to run exactly two steps.")


## PHASE 4 — Manual training, visualization, checkpoints, and resume

**Cell order:** cell 27 only defines the training loop. Execute cells 28–31 to define all its helpers before starting a run. The explicit start switch is at the bottom of cell 31 and remains **False** in the delivered notebook. The default stop is epoch 100. Completed epochs resume only from tagged reproduction checkpoints; official and pilot checkpoints are rejected.


### 27. Visible tqdm training loop

Define the loop without starting it. Each epoch has exactly 300 steps. tqdm displays Epoch X/Y, Step X/300, G_total, D_A, D_B, weighted Cycle/Identity totals, LR, elapsed time, and ETA. Fresh training reinitializes from seed 171, independently of preview models or smoke testing. All logs and manifests are kept in unique reproduction run/segment folders.


In [ ]:
EPOCH_FIELDS = ["epoch", "global_step", "learning_rate"] + LOSS_KEYS + [
    "generator_gradient_norm", "discriminator_gradient_norm", "NaN_count", "Inf_count",
    "images_per_second", "epoch_seconds", "elapsed_minutes", "peak_gpu_memory_GB",
]
STEP_FIELDS = ["epoch", "step", "global_step", "learning_rate"] + LOSS_KEYS + [
    "generator_gradient_norm", "discriminator_gradient_norm", "NaN_count", "Inf_count",
]

def run_repro_training(stop_epoch=100, resume_checkpoint=None):
    if not 1 <= stop_epoch <= CONFIG["epochs"]:
        raise ValueError("Stop epoch must be within the unchanged 1..200 schedule.")
    started = time.perf_counter()
    seed_all(SEED)
    models = build_models(DEVICE)
    optimizers, gp, dp = make_optimizers(models)
    schedulers = make_schedulers(optimizers)
    first_epoch, global_step, elapsed_before = 1, 0, 0.0
    run_id = new_run_id()
    fixed = {domain: list(identifiers) for domain, identifiers in FIXED_IDENTIFIERS_REPRO.items()}
    pending = False
    if resume_checkpoint is not None:
        checkpoint = load_repro_resume(resume_checkpoint, models, optimizers, schedulers)
        first_epoch = checkpoint["epoch"] + 1
        global_step = checkpoint["global_step"]
        run_id = checkpoint["run_id"]
        fixed = checkpoint["fixed_samples"]
        elapsed_before = checkpoint["elapsed_seconds"]
        pending = checkpoint["scheduler_pending_step"]
    if first_epoch > stop_epoch:
        print(f"The reproduction checkpoint already reached epoch {stop_epoch}; no training starts.")
        return {"run_id": run_id, "epoch": first_epoch - 1, "global_step": global_step}
    if pending:
        for name, scheduler in schedulers.items():
            optimizers[name]._opt_called = True
            scheduler.step()

    checkpoint_dir = require_repro_path(CHECKPOINTS_REPRO / run_id, CHECKPOINTS_REPRO)
    logs_dir = require_repro_path(LOGS_REPRO / run_id, LOGS_REPRO)
    outputs_dir = require_repro_path(OUTPUTS_REPRO / run_id, OUTPUTS_REPRO)
    if resume_checkpoint is None:
        for directory in (checkpoint_dir, logs_dir, outputs_dir):
            directory.mkdir(parents=True, exist_ok=False)
    else:
        for directory in (checkpoint_dir, logs_dir, outputs_dir):
            directory.mkdir(parents=True, exist_ok=True)
        latest = checkpoint_dir / "latest.pt"
        if latest.exists():
            latest_state = torch.load(latest, map_location="cpu", weights_only=False)
            if latest_state["epoch"] > first_epoch - 1:
                raise ValueError("The selected checkpoint is older than this run's latest checkpoint.")
            del latest_state

    # Check before any optimization that the requested future artifacts are unused.
    for future_epoch in range(first_epoch, stop_epoch + 1):
        if (checkpoint_dir / f"epoch_{future_epoch:03d}.pt").exists():
            raise FileExistsError("Refusing to replay an epoch with an existing permanent checkpoint.")
        if (outputs_dir / "fixed_samples" / f"epoch_{future_epoch:03d}").exists():
            raise FileExistsError("Refusing to overwrite an existing fixed-sample set.")
    segment_id = new_run_id()
    manifest_dir = require_repro_path(outputs_dir / "segments" / segment_id / "manifests")
    write_repro_json(manifest_dir / "environment.json", {
        **ENVIRONMENT, "seed": SEED, "sid4": SID4, "configuration": CONFIG,
        "run_id": run_id, "first_epoch": first_epoch, "stop_epoch": stop_epoch,
    })
    write_repro_json(manifest_dir / "fixed_samples.json", fixed)
    write_repro_json(manifest_dir / "dataset.json", {
        "A": [path.relative_to(ROOT).as_posix() for path in PATHS_A],
        "B": [path.relative_to(ROOT).as_posix() for path in PATHS_B],
    })
    csv_path = require_repro_path(logs_dir / f"{segment_id}_epochs.csv", LOGS_REPRO)
    step_path = require_repro_path(logs_dir / f"{segment_id}_steps.csv", LOGS_REPRO)
    raw_path = require_repro_path(logs_dir / f"{segment_id}_raw.log", LOGS_REPRO)
    history = []
    epoch, index = first_epoch, 0
    with csv_path.open("x", newline="", encoding="utf-8") as csv_file, \
         step_path.open("x", newline="", encoding="utf-8") as step_file, \
         raw_path.open("x", encoding="utf-8", buffering=1) as raw:
        epoch_writer = csv.DictWriter(csv_file, fieldnames=EPOCH_FIELDS)
        step_writer = csv.DictWriter(step_file, fieldnames=STEP_FIELDS)
        epoch_writer.writeheader()
        step_writer.writeheader()
        raw.write(f"REPRO seed={SEED} sid4={SID4} run={run_id}\nConfiguration: {CONFIG}\n")
        try:
            for epoch in range(first_epoch, stop_epoch + 1):
                index = 0
                assert all(abs(opt.param_groups[0]["lr"] - epoch_lr(epoch)) < 1e-12
                           for opt in optimizers.values())
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize(DEVICE)
                    torch.cuda.reset_peak_memory_stats(DEVICE)
                epoch_started = time.perf_counter()
                dataset = EpochDataset(PATHS_A, PATHS_B)
                assert len(dataset) == 300
                assert sorted(a for a, _ in dataset.pairs) == list(range(300))
                write_repro_json(manifest_dir / f"epoch_{epoch:03d}_pairs.json", dataset.manifest())
                loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
                sums = {name: 0.0 for name in LOSS_KEYS + [
                    "generator_gradient_norm", "discriminator_gradient_norm", "NaN_count", "Inf_count"]}
                lr = optimizers["G"].param_groups[0]["lr"]
                with tqdm(total=300, desc=f"Epoch {epoch}/{stop_epoch}",
                          bar_format="{desc} | Step {n_fmt}/300 |{bar}| [{elapsed}<{remaining}] {postfix}") as progress:
                    for index, (a, b) in enumerate(loader, 1):
                        values, tensors = train_step(models, optimizers, gp, dp, a.to(DEVICE), b.to(DEVICE))
                        del tensors
                        global_step += 1
                        for name, value in values.items():
                            sums[name] += value
                        step_writer.writerow({
                            "epoch": epoch, "step": index, "global_step": global_step,
                            "learning_rate": lr, **values,
                        })
                        raw.write(f"epoch={epoch} step={index} global_step={global_step} lr={lr} values={values}\n")
                        epoch_elapsed = time.perf_counter() - epoch_started
                        remaining_steps = (300 - index) + (stop_epoch - epoch) * 300
                        eta_seconds = epoch_elapsed / index * remaining_steps
                        progress.set_postfix({
                            "G_total": f"{values['G_total']:.4f}",
                            "D_A": f"{values['D_A']:.4f}", "D_B": f"{values['D_B']:.4f}",
                            "Cycle": f"{values['cycle_A'] + values['cycle_B']:.4f}",
                            "Identity": f"{values['identity_A'] + values['identity_B']:.4f}",
                            "LR": f"{lr:.8f}",
                            "elapsed": format_duration(elapsed_before + time.perf_counter() - started),
                            "ETA": format_duration(eta_seconds),
                        }, refresh=False)
                        progress.update(1)
                        if index % 50 == 0:
                            step_file.flush()
                assert index == 300 and global_step == epoch * 300
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize(DEVICE)
                seconds = time.perf_counter() - epoch_started
                row = {name: value if name in ("NaN_count", "Inf_count") else value / 300
                       for name, value in sums.items()}
                row.update(
                    epoch=epoch, global_step=global_step, learning_rate=lr,
                    images_per_second=600 / seconds, epoch_seconds=seconds,
                    elapsed_minutes=(elapsed_before + time.perf_counter() - started) / 60,
                    peak_gpu_memory_GB=(torch.cuda.max_memory_allocated(DEVICE) / 1024**3
                                        if DEVICE.type == "cuda" else 0.0),
                )
                epoch_writer.writerow(row)
                csv_file.flush()
                step_file.flush()
                raw.write(f"EPOCH SUMMARY {row}\n")
                history.append(row)
                live_epoch_summary(history)
                if epoch % 5 == 0:
                    visualize_fixed_samples_repro(models, fixed, outputs_dir / "fixed_samples" / f"epoch_{epoch:03d}")
                pending = epoch == stop_epoch
                if not pending:
                    for scheduler in schedulers.values():
                        scheduler.step()
                elapsed = elapsed_before + time.perf_counter() - started
                metadata = {"run_id": run_id, "scheduler_pending_step": pending, "stop_epoch": stop_epoch}
                save_repro_checkpoint(checkpoint_dir / "latest.pt", models, optimizers, schedulers,
                                      epoch, global_step, fixed, elapsed, metadata)
                if epoch % 10 == 0:
                    save_repro_checkpoint(checkpoint_dir / f"epoch_{epoch:03d}.pt",
                                          models, optimizers, schedulers, epoch, global_step, fixed, elapsed, metadata)
            summary = {
                "run_id": run_id, "first_epoch": first_epoch, "stop_epoch": stop_epoch,
                "global_step": global_step, "elapsed_seconds": elapsed_before + time.perf_counter() - started,
                "final_epoch": history[-1], "NaN_total": sum(row["NaN_count"] for row in history),
                "Inf_total": sum(row["Inf_count"] for row in history), "configuration": CONFIG,
            }
            write_repro_json(outputs_dir / "segments" / segment_id / "summary.json", summary)
            raw.write(f"COMPLETED {summary}\n")
            print(f"Reproduction stopped cleanly at epoch {stop_epoch}; step {global_step}.")
            return summary
        except BaseException as error:
            raw.write(traceback.format_exc())
            csv_file.flush()
            step_file.flush()
            diagnostics = getattr(error, "diagnostics", {})
            metadata = {
                "run_id": run_id, "diagnostic_only": True, "failed_epoch": epoch,
                "failed_step": index, "error": repr(error), "diagnostics": diagnostics,
                "gradients": {
                    name: {key: parameter.grad.detach().cpu() for key, parameter in model.named_parameters()
                           if parameter.grad is not None}
                    for name, model in models.items()
                },
            }
            save_repro_checkpoint(checkpoint_dir / f"diagnostic_{segment_id}_epoch_{epoch:03d}_step_{index:03d}.pt",
                                  models, optimizers, schedulers, epoch - 1, global_step, fixed,
                                  elapsed_before + time.perf_counter() - started, metadata)
            raw.write("Diagnostic saved. Training stopped. No automatic continuation.\n")
            raise

print("Visible tqdm training loop is defined; no training has started. Complete helper cells 28–31 next.")


### 28. Live epoch summaries

Define live tables and timing labels. Epoch CSVs contain all individual losses, gradient norms, LR, NaN/Inf totals, throughput, elapsed time, and peak GPU memory. Raw logs are append-only within an exclusively created segment.


In [ ]:
def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def live_epoch_summary(history):
    latest = history[-1]
    print(
        f"Epoch {latest['epoch']} completed | step={latest['global_step']} | "
        f"G_total={latest['G_total']:.4f} | D_A={latest['D_A']:.4f} | D_B={latest['D_B']:.4f} | "
        f"Cycle={latest['cycle_A'] + latest['cycle_B']:.4f} | "
        f"Identity={latest['identity_A'] + latest['identity_B']:.4f} | "
        f"LR={latest['learning_rate']:.8f} | NaN={int(latest['NaN_count'])} | Inf={int(latest['Inf_count'])}"
    )
    columns = ["epoch", "global_step", "G_total", "D_A", "D_B", "cycle_A", "cycle_B",
               "identity_A", "identity_B", "learning_rate", "generator_gradient_norm",
               "discriminator_gradient_norm", "images_per_second", "epoch_seconds",
               "elapsed_minutes", "peak_gpu_memory_GB", "NaN_count", "Inf_count"]
    display(pd.DataFrame(history)[columns].tail(10))


### 29. Fixed sample visualization

Select four identifiers per domain deterministically before training. Every five epochs, save direct RGB translations plus a comparison grid under outputs_repro. Resume uses the identifiers stored in the reproduction checkpoint.


In [ ]:
fixed_rng = random.Random(SEED)
FIXED_IDENTIFIERS_REPRO = {
    domain: [path.relative_to(ROOT).as_posix()
             for path in fixed_rng.sample(paths, CONFIG["fixed_sample_count"])]
    for domain, paths in [("A", PATHS_A), ("B", PATHS_B)]
}
print("Fixed identifiers, selected before training:", FIXED_IDENTIFIERS_REPRO)

@torch.no_grad()
def visualize_fixed_samples_repro(models, fixed, folder):
    folder = require_repro_path(folder, OUTPUTS_REPRO)
    if folder.exists():
        raise FileExistsError(f"Refusing to overwrite fixed samples: {folder}")
    folder.mkdir(parents=True, exist_ok=False)
    modes = {name: models[name].training for name in ("G_A2B", "G_B2A")}
    for name in modes:
        models[name].eval()
    figure, axes = plt.subplots(4, CONFIG["fixed_sample_count"], figsize=(14, 12), squeeze=False)
    try:
        for domain, direction, base_row in [("A", "A2B", 0), ("B", "B2A", 2)]:
            for column, identifier in enumerate(fixed[domain]):
                path = ROOT / identifier
                image = load_image(path, training=False).unsqueeze(0).to(DEVICE)
                translated = models["G_" + direction](image)
                save_repro_image(translated, folder / direction / (path.stem + ".png"))
                for row, tensor, title in [
                    (base_row, image, f"Real {domain}: {path.name}"),
                    (base_row + 1, translated, f"{direction}: {path.name}"),
                ]:
                    display_array = ((tensor.detach().cpu().squeeze(0) + 1) / 2).clamp(0, 1)
                    axes[row, column].imshow(display_array.permute(1, 2, 0).numpy())
                    axes[row, column].set_title(title)
                    axes[row, column].axis("off")
        figure.tight_layout()
        figure.savefig(require_repro_path(folder / "fixed_grid.png", OUTPUTS_REPRO), dpi=140)
        plt.show()
    finally:
        plt.close(figure)
        for name, was_training in modes.items():
            models[name].train(was_training)


### 30. Checkpoint saving

Save latest.pt every completed epoch and permanent checkpoints every ten epochs, including 050 and 100. Every checkpoint contains four model states, optimizer/scheduler states, configuration, seed/SID4, fixed identifiers, and RNG states. Only the active reproduction run's latest.pt can be replaced.


In [ ]:
def save_repro_checkpoint(path, models, optimizers, schedulers, epoch, global_step, fixed, elapsed, metadata):
    path = require_repro_path(path, CHECKPOINTS_REPRO)
    if path.exists() and path.name != "latest.pt":
        raise FileExistsError(f"Refusing to overwrite a permanent checkpoint: {path}")
    payload = {
        "run_kind": "repro", "seed": SEED, "sid4": SID4,
        "epoch": epoch, "global_step": global_step, "configuration": CONFIG,
        "models": {name: model.state_dict() for name, model in models.items()},
        "optimizers": {name: optimizer.state_dict() for name, optimizer in optimizers.items()},
        "schedulers": {name: scheduler.state_dict() for name, scheduler in schedulers.items()},
        "random_states": random_states(), "fixed_samples": fixed, "elapsed_seconds": elapsed,
        **metadata,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = require_repro_path(path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp"), CHECKPOINTS_REPRO)
    torch.save(payload, temporary)
    temporary.replace(path)

print("Reproduction checkpoint saver defined. No checkpoint has been saved by this cell.")


In [ ]:
# Free preview-only GPU objects before official reproduction training
for name in [
    "preview_models",
    "preview_optimizers",
    "preview_schedulers",
    "preview_gp",
    "preview_dp",
]:
    if name in globals():
        del globals()[name]

import gc
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("Preview GPU objects cleared.")

### 31. Resume support and manual start — disabled by default

Resume only a trusted, complete-epoch checkpoint under checkpoints_repro. Official, pilot, smoke, and diagnostic checkpoints are rejected. RNG and scheduler state are restored; the run remains bounded by STOP_EPOCH_REPRO=100. To start manually after inspecting all cells, set RUN_TRAINING_REPRO=True in this cell. For fresh training leave RESUME_CHECKPOINT_REPRO=None. No shell or command-line training program is launched.


In [ ]:
def load_repro_resume(path, models, optimizers, schedulers):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    path = require_repro_path(path, CHECKPOINTS_REPRO)
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if checkpoint.get("run_kind") != "repro" or checkpoint.get("diagnostic_only"):
        raise ValueError("Only complete-epoch reproduction checkpoints may resume.")
    if checkpoint["configuration"] != CONFIG or checkpoint["seed"] != SEED or checkpoint["sid4"] != SID4:
        raise ValueError("Checkpoint seed, SID4, or locked configuration differs.")
    if checkpoint["global_step"] != checkpoint["epoch"] * 300:
        raise ValueError("Checkpoint is not at an exact 300-step epoch boundary.")
    if path.parent != require_repro_path(CHECKPOINTS_REPRO / checkpoint["run_id"], CHECKPOINTS_REPRO):
        raise ValueError("Checkpoint run identifier does not match its reproduction directory.")
    for name, model in models.items():
        model.load_state_dict(checkpoint["models"][name])
    for name, optimizer in optimizers.items():
        optimizer.load_state_dict(checkpoint["optimizers"][name])
        schedulers[name].load_state_dict(checkpoint["schedulers"][name])
    restore_random(checkpoint["random_states"])
    return checkpoint

# MANUAL START CONTROLS. Delivered disabled. 
RUN_TRAINING_REPRO = False
STOP_EPOCH_REPRO = 100
RESUME_CHECKPOINT_REPRO = None
# Example after a manual run: "checkpoints_repro/<run_id>/latest.pt"

if RUN_TRAINING_REPRO:
    REPRO_TRAINING_RESULT = run_repro_training(
        stop_epoch=STOP_EPOCH_REPRO,
        resume_checkpoint=RESUME_CHECKPOINT_REPRO,
    )
else:
    print("Training is disabled. Notebook setup is ready for manual inspection.")
    print("No reproduction epoch has started; no official checkpoint, log, output, or evaluation is modified.")


In [ ]:
# ============================================================
# CONTINUE REPRODUCTION TRAINING: EPOCH 101 -> 150
# ============================================================

from pathlib import Path

# Find the epoch-100 checkpoint from this reproduction run
epoch100_checkpoints = sorted(CHECKPOINTS_REPRO.rglob("epoch_100.pt"))

assert len(epoch100_checkpoints) > 0, "epoch_100.pt not found."

RESUME_CHECKPOINT_REPRO = epoch100_checkpoints[-1]

RUN_TRAINING_REPRO = False
STOP_EPOCH_REPRO = 150

print("Resuming from:", RESUME_CHECKPOINT_REPRO)
print("Target stop epoch:", STOP_EPOCH_REPRO)
print("Expected additional epochs: 50")
print("Expected additional steps:", 50 * 300)

REPRO_TRAINING_RESULT_150 = run_repro_training(
    stop_epoch=STOP_EPOCH_REPRO,
    resume_checkpoint=RESUME_CHECKPOINT_REPRO,
)

In [ ]:
# ============================================================
# EVALUATE REPRO CHECKPOINT: EPOCH 150
# ============================================================

from pathlib import Path
import torch
from tqdm.auto import tqdm

epoch150_files = sorted(CHECKPOINTS_REPRO.rglob("epoch_150.pt"))

assert len(epoch150_files) > 0, "epoch_150.pt not found"

EPOCH150_CHECKPOINT = epoch150_files[-1]

print("Using checkpoint:")
print(EPOCH150_CHECKPOINT)

checkpoint_150 = torch.load(
    EPOCH150_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Epoch:", checkpoint_150["epoch"])
print("Global step:", checkpoint_150["global_step"])
print("Run ID:", checkpoint_150["run_id"])

assert checkpoint_150["epoch"] == 150
assert checkpoint_150["global_step"] == 45000

In [ ]:
eval_models_150 = build_models(DEVICE)

eval_models_150["G_A2B"].load_state_dict(
    checkpoint_150["models"]["G_A2B"]
)

eval_models_150["G_B2A"].load_state_dict(
    checkpoint_150["models"]["G_B2A"]
)

eval_models_150["G_A2B"].eval()
eval_models_150["G_B2A"].eval()

print("Epoch-150 generators loaded successfully.")

In [ ]:
EVAL150_ROOT = OUTPUTS_REPRO / "evaluation" / "epoch_150"

PRED_A2B_150 = EVAL150_ROOT / "pred_A2B"
PRED_B2A_150 = EVAL150_ROOT / "pred_B2A"

if EVAL150_ROOT.exists():
    raise FileExistsError(
        f"{EVAL150_ROOT} already exists. "
        "Do not overwrite a previous evaluation."
    )

PRED_A2B_150.mkdir(parents=True)
PRED_B2A_150.mkdir(parents=True)

print("A2B:", PRED_A2B_150)
print("B2A:", PRED_B2A_150)

In [ ]:
@torch.no_grad()
def generate_epoch150_predictions():

    G_A2B = eval_models_150["G_A2B"]
    G_B2A = eval_models_150["G_B2A"]

    # A2B: Monet -> Photo
    for path in tqdm(PATHS_A, desc="Epoch150 A2B Monet -> Photo"):
        x = load_image(path, training=False).unsqueeze(0).to(DEVICE)

        fake_B = G_A2B(x)

        save_repro_image(
            fake_B,
            PRED_A2B_150 / f"{path.stem}.jpg"
        )

    # B2A: Photo -> Monet
    for path in tqdm(PATHS_B, desc="Epoch150 B2A Photo -> Monet"):
        x = load_image(path, training=False).unsqueeze(0).to(DEVICE)

        fake_A = G_B2A(x)

        save_repro_image(
            fake_A,
            PRED_B2A_150 / f"{path.stem}.jpg"
        )

    print("Inference finished.")


generate_epoch150_predictions()

In [ ]:
a2b_files = sorted(PRED_A2B_150.glob("*.jpg"))
b2a_files = sorted(PRED_B2A_150.glob("*.jpg"))

print("A2B:", len(a2b_files))
print("B2A:", len(b2a_files))

assert len(a2b_files) == 300
assert len(b2a_files) == 7038

# Check a few files
from PIL import Image

for path in a2b_files[:3] + b2a_files[:3]:
    with Image.open(path) as img:
        assert img.size == (256, 256)
        assert img.mode == "RGB"

print("✅ All counts and image checks passed.")

In [ ]:
print(PRED_A2B_150)

In [ ]:
print(PRED_B2A_150)

In [ ]:
METRICS_150 = EVAL150_ROOT / "metrics"
METRICS_150.mkdir(parents=True, exist_ok=True)

print(METRICS_150)

In [ ]:
print(PRED_A2B_150)
print(PRED_B2A_150)

# Epoch 150 — Official Instructor Evaluation


In [ ]:
# Read saved evaluation evidence only. This cell does not train or evaluate models.
import csv
from pathlib import Path

REPORT150_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data/monet_jpg").is_dir()
    and (candidate / "outputs_repro/evaluation/epoch_150/metrics").is_dir()
)
REPORT150_METRICS = REPORT150_ROOT / "outputs_repro/evaluation/epoch_150/metrics"
REPORT100_METRICS = REPORT150_ROOT / "outputs_repro/evaluation/epoch_100/metrics"

with (REPORT150_METRICS / "evaluation_results.csv").open(newline="", encoding="utf-8") as report_file:
    REPORT150_ROWS = list(csv.DictReader(report_file))
assert len(REPORT150_ROWS) == 1 and int(REPORT150_ROWS[0]["checkpoint_epoch"]) == 150
REPORT150_ROW = REPORT150_ROWS[0]
REPORT150_SCORES = {
    name: float(REPORT150_ROW[name])
    for name in ("FID_A2B", "MiFID_A2B", "FID_B2A", "MiFID_B2A", "submission_FID", "submission_MiFID")
}
with (REPORT150_METRICS / "submission.csv").open(newline="", encoding="utf-8") as report_file:
    REPORT150_SUBMISSION = list(csv.DictReader(report_file))
assert len(REPORT150_SUBMISSION) == 1
assert float(REPORT150_SUBMISSION[0]["FID"]) == REPORT150_SCORES["submission_FID"]
assert float(REPORT150_SUBMISSION[0]["MiFID"]) == REPORT150_SCORES["submission_MiFID"]

with (REPORT100_METRICS / "submission.csv").open(newline="", encoding="utf-8") as report_file:
    REPORT100_SUBMISSION = list(csv.DictReader(report_file))
assert len(REPORT100_SUBMISSION) == 1
REPORT100_FID = float(REPORT100_SUBMISSION[0]["FID"])
REPORT100_MIFID = float(REPORT100_SUBMISSION[0]["MiFID"])
assert abs(REPORT100_FID - 112.4340777163) < 1e-9
assert abs(REPORT100_MIFID - 0.4231333733) < 1e-9

print("============================================================")
print("EPOCH 150 — OFFICIAL EVALUATION RESULTS")
print("============================================================")
print()
print(f"FID_A2B        : {REPORT150_SCORES['FID_A2B']}")
print(f"MiFID_A2B      : {REPORT150_SCORES['MiFID_A2B']}")
print()
print(f"FID_B2A        : {REPORT150_SCORES['FID_B2A']}")
print(f"MiFID_B2A      : {REPORT150_SCORES['MiFID_B2A']}")
print()
print(f"Submission FID : {REPORT150_SCORES['submission_FID']}")
print(f"Submission MiFID: {REPORT150_SCORES['submission_MiFID']}")
print()
print("Checkpoint     : epoch_150")
print("Global step    : 45000")
print()
print("============================================================")
print()
print("Metric               Epoch 100          Epoch 150")
print("---------------------------------------------------")
print(f"{'Submission FID':<21}{REPORT100_FID:<19.10f}{REPORT150_SCORES['submission_FID']:.10f}")
print(f"{'Submission MiFID':<21}{REPORT100_MIFID:<19.10f}{REPORT150_SCORES['submission_MiFID']:.10f}")
print()
print("Numerical differences (Epoch 150 − Epoch 100):")
print(f"Submission FID  : {REPORT150_SCORES['submission_FID'] - REPORT100_FID:+.12f}")
print(f"Submission MiFID: {REPORT150_SCORES['submission_MiFID'] - REPORT100_MIFID:+.12f}")


# Final Epoch-150 Evaluation Summary


In [ ]:
# Read saved final metrics only. This cell never trains or evaluates a model.
from pathlib import Path
import csv

FINAL150_PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "outputs_repro/final_metrics_epoch150/automated_metrics_epoch150.csv").is_file()
)
FINAL150_DIR = FINAL150_PROJECT / "outputs_repro/final_metrics_epoch150"
with (FINAL150_DIR / "automated_metrics_epoch150.csv").open(newline="", encoding="utf-8") as final_file:
    FINAL150_ROWS = list(csv.DictReader(final_file))
assert len(FINAL150_ROWS) == 1
FINAL150 = FINAL150_ROWS[0]
assert int(FINAL150["epoch"]) == 150 and int(FINAL150["global_step"]) == 45000

def final150_value(key):
    return FINAL150[key] if FINAL150[key] else "UNAVAILABLE"

def final150_line(label, key):
    print(f"{label:<31}: {final150_value(key)}")

def final150_mean_std(label, prefix):
    print(f"{label:<31}: {final150_value(prefix+'_mean')} / {final150_value(prefix+'_std')} (mean/std)")

print("============================================================")
print("FINAL EPOCH 150 — TASK 3 METRICS")
print("============================================================")
print("Checkpoint:", FINAL150["checkpoint"])
print("Epoch: 150 | Global step: 45000 | Seed: 171")
print()
print("Official FID / MiFID")
for label, key in [("FID_A2B","FID_A2B"),("MiFID_A2B","MiFID_A2B"),
                   ("FID_B2A","FID_B2A"),("MiFID_B2A","MiFID_B2A"),
                   ("Submission FID","submission_FID"),("Submission MiFID","submission_MiFID")]:
    final150_line(label,key)
print()
print("Remaining automated metrics")
final150_mean_std("KID A2B","KID_A2B")
final150_mean_std("KID B2A","KID_B2A")
for label,key in [("Generative Precision A2B","generative_precision_A2B"),
                  ("Generative Recall A2B","generative_recall_A2B"),
                  ("Generative Precision B2A","generative_precision_B2A"),
                  ("Generative Recall B2A","generative_recall_B2A"),
                  ("Cycle L1 A [0,1]","cycle_L1_A_0_1"),
                  ("Cycle L1 B [0,1]","cycle_L1_B_0_1")]:
    final150_line(label,key)
final150_mean_std("LPIPS A","LPIPS_A")
final150_mean_std("LPIPS B","LPIPS_B")
final150_mean_std("Content Cosine A2B","content_cosine_A2B")
final150_mean_std("Content Cosine B2A","content_cosine_B2A")
print()
print("Training stability — existing epoch-150 log")
for label,key in [("G_total","G_total"),("D_A","D_A"),("D_B","D_B"),
                  ("Generator gradient norm","generator_gradient_norm"),
                  ("Discriminator gradient norm","discriminator_gradient_norm"),
                  ("NaN count","NaN_count"),("Inf count","Inf_count")]:
    final150_line(label,key)
print()
print("Efficiency — existing training records")
for label,key in [("Parameter count (four models)","parameter_count_total"),
                  ("Training time (logged minutes)","training_time_minutes"),
                  ("Images/sec (epoch 150)","images_per_second"),
                  ("Peak GPU GiB (epoch 150)","peak_gpu_memory_GB"),
                  ("Peak GPU GiB (entire history)","history_peak_gpu_memory_GiB")]:
    final150_line(label,key)
print()
print("Still pending:")
print("- 30-image blinded human audit with two raters: PENDING")
print("- Cohen's kappa / percent agreement: PENDING")
print("- Final Kaggle public/private score and leaderboard rank: PENDING")
print()
print("Evidence:")
print("outputs_repro/final_metrics_epoch150/automated_metrics_epoch150.csv")
print("outputs_repro/final_metrics_epoch150/full_metrics_report.csv")
print("notebooks/evaluate_cyclegan.ipynb")
print("============================================================")
